In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import xgboost as xgb

In [11]:
data = pd.read_csv('../datasets/titanic.csv')
data.columns = [col.lower() for col in data.columns]
data

,passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [34]:
# data
data['title'] = data['name'].apply(lambda x: x.split(',')[1].split('.')[0].strip())

# Group rare titles
title_mapping = {
    'Mr': 'Mr',
    'Miss': 'Miss',
    'Mrs': 'Mrs',
    'Master': 'Master',
    'Dr': 'Rare',
    'Rev': 'Rare',
    'Col': 'Rare',
    'Major': 'Rare',
    'Mlle': 'Miss',
    'Mme': 'Mrs',
    'Ms': 'Miss',
    'Lady': 'Rare',
    'Sir': 'Rare',
    'Countess': 'Rare',
    'Jonkheer': 'Rare',
    'Don': 'Rare'
}
data['title'] = data['title'].map(title_mapping)
data['family_size'] = data['sibsp'] + data['parch'] + 1
data['is_alone'] =  (data['family_size'] == 1).astype(int)
data['cabin_prefix'] = data['cabin'].apply(lambda x: str(x)[0] if pd.notna(x) else 'Unknown') # cabin prefix
data['age_group'] = pd.cut(data['age'], bins=[0, 12, 18, 30, 50, 100], labels=['Child', 'Teen', 'Young Adult', 'Adult', 'Elder'])
data['fare_group'] = pd.cut(data['fare'], bins=[0, 10, 25, 50, 100, 600],labels=['Low', 'Medium', 'High', 'Very High', 'Luxury'])

features = [
    'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
    'embarked', 'title', 'family_size', 'is_alone', 'cabin_prefix'
]

X = data[features]
y = data['survived']

# missing values
X_prepared = X.copy()
X_prepared['age'].fillna(X_prepared['age'].median(), inplace=True)
X_prepared['fare'].fillna(X_prepared['fare'].median(), inplace=True)
X_prepared['embarked'].fillna(X_prepared['embarked'].mode()[0], inplace=True)

# encode categorical features
le = LabelEncoder()
categorical_cols = ['sex', 'embarked', 'title', 'cabin_prefix']
for col in categorical_cols:
    X_prepared[col] = le.fit_transform(X_prepared[col].astype(str))

# data split
X_train, X_test, y_train, y_test = train_test_split(X_prepared, y, test_size=0.2, random_state=42, stratify=y)

# model (XGBoost doesn't require feature scaling!)
model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    objective='binary:logistic', # default
)
model.fit(X_train, y_train)

# predict
y_pred = model.predict(X_test)
y_pred_train = model.predict(X_train)
y_pred_proba = model.predict_proba(X_test)

C:\Users\rah\AppData\Local\Temp\ipykernel_20700\3592936092.py:40: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_prepared['age'].fillna(X_prepared['age'].median(), inplace=True)
C:\Users\rah\AppData\Local\Temp\ipykernel_20700\3592936092.py:41: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as 

In [35]:
target_names = ['not survived', 'survived']

print("\n" + "="*60)
print("MODEL EVALUATION")
print("="*60)

# accuracy
test_accuracy = accuracy_score(y_test, y_pred)
train_accuracy = accuracy_score(y_train, y_pred_train)
gap = train_accuracy - test_accuracy
print(f"Training Accuracy: {train_accuracy:.2%}")
print(f"Test Accuracy: {test_accuracy:.2%}")
print(f"Overfitting Gap: {gap:.4f}")


if gap < 0.01:
    print("✅ NO OVERFITTING (gap < 1%)")
    status = "Balanced"
elif gap < 0.02:
    print("✅ MINIMAL OVERFITTING (1-2%)")
    status = "Slight Overfitting"
elif gap < 0.03:
    print("⚠️  MILD OVERFITTING (2-3%)")
    status = "Mild Overfitting"
elif gap < 0.05:
    print("⚠️  MODERATE OVERFITTING (3-5%)")
    status = "Moderate Overfitting"
elif gap < 0.10:
    print("🔴 SIGNIFICANT OVERFITTING (5-10%)")
    status = "Significant Overfitting"
else:
    print("🔴 SEVERE OVERFITTING (>10%)")
    status = "Severe Overfitting"

if train_accuracy < 0.70 and test_accuracy < 0.70:
    print("📉 UNDERFITTING DETECTED: Model is too simple")
elif train_accuracy < 0.75:
    print("⚠️  Possible underfitting (training accuracy < 75%)")
else:
    print("✅ No underfitting detected")

# detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

# confusion matrix
cm = confusion_matrix(y_test, y_pred)
correct_predictions = np.trace(cm)
total_predictions = np.sum(cm)
accuracy = correct_predictions / total_predictions

print("\nConfusion Matrix:")
print(cm)


print(f"Correct predictions: {correct_predictions}")
print(f"Total predictions: {total_predictions}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy*100:.2f}%")
print()

for i in range(cm.shape[0]):
    class_correct = cm[i, i]
    class_total = np.sum(cm[i, :])
    accuracy = class_correct / class_total
    print(f"{target_names[i]} Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")


cv_scores = cross_val_score(model, X_train, y_train, cv=5)

print("\n" + "="*60)
print("CROSS-VALIDATION")
print("="*60)
print(f"CV Scores: {cv_scores}")
print(f"CV Mean: {cv_scores.mean():.4f}")
print(f"CV Std: {cv_scores.std():.4f}")


MODEL EVALUATION
Training Accuracy: 92.84%
Test Accuracy: 79.33%
Overfitting Gap: 0.1351
🔴 SEVERE OVERFITTING (>10%)
✅ No underfitting detected

Classification Report:
              precision    recall  f1-score   support

not survived       0.80      0.88      0.84       110
    survived       0.78      0.65      0.71        69

    accuracy                           0.79       179
   macro avg       0.79      0.77      0.77       179
weighted avg       0.79      0.79      0.79       179


Confusion Matrix:
[[97 13]
 [24 45]]
Correct predictions: 142
Total predictions: 179
Accuracy: 0.7933
Accuracy: 79.33%

not survived Accuracy: 0.8818 (88.18%)
survived Accuracy: 0.6522 (65.22%)

CROSS-VALIDATION
CV Scores: [0.7972028  0.76223776 0.82394366 0.84507042 0.83098592]
CV Mean: 0.8119
CV Std: 0.0293


In [25]:
feature_importance = pd.DataFrame({
    'feature': X_prepared.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
feature_importance

,feature,importance
1,sex,0.602553
0,pclass,0.131151
10,cabin_prefix,0.078866
7,title,0.033483
8,family_size,0.031517
2,age,0.029162
5,fare,0.027896
6,embarked,0.026803
4,parch,0.019339
3,sibsp,0.019230


In [53]:
param_grid = {
    'max_depth': [2, 3, 4],
    'min_child_weight': [3, 5, 7],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'reg_alpha': [0.1, 0.5, 1.0],
    'reg_lambda': [0.5, 1.0, 2.0],
    'n_estimators': [1, 20, 40, 50, 100, 150, 200, 1000]
}

grid_search = GridSearchCV(
    estimator=xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.05,
        random_state=42
    ),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"Best Params: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_
train_acc_best = best_model.score(X_train, y_train)
test_acc_best = best_model.score(X_test, y_test)

Fitting 5 folds for each of 5832 candidates, totalling 29160 fits
Best Params: {'colsample_bytree': 0.6, 'max_depth': 2, 'min_child_weight': 3, 'n_estimators': 1000, 'reg_alpha': 1.0, 'reg_lambda': 2.0, 'subsample': 0.6}
Best CV Score: 0.8358


In [60]:
# model is too complex for this small dataset so we remove some of columns
features = [
    'pclass', 'sex', 'age', 'fare',
    'embarked','title', 'family_size', 'cabin_prefix'
]

X = data[features]
y = data['survived']

# missing values
X_prepared = X.copy()
X_prepared['age'].fillna(X_prepared['age'].median(), inplace=True)
X_prepared['fare'].fillna(X_prepared['fare'].median(), inplace=True)
X_prepared['embarked'].fillna(X_prepared['embarked'].mode()[0], inplace=True)

# encode categorical features
le = LabelEncoder()
categorical_cols = ['sex', 'embarked', 'title', 'cabin_prefix']
for col in categorical_cols:
    X_prepared[col] = le.fit_transform(X_prepared[col].astype(str))

# data split
X_train, X_test, y_train, y_test = train_test_split(X_prepared, y, test_size=0.2, random_state=42, stratify=y)

# parameters
params = {
    'n_estimators': 50, # changed
    'learning_rate': 0.1,
    'max_depth': 2, # changed
    'random_state': 42,
    'objective': 'binary:logistic',
    'min_child_weight': 5 # added
}

# model (XGBoost doesn't require feature scaling!)
model = xgb.XGBClassifier(**params)
model.fit(X_train, y_train)

# predict
y_pred = model.predict(X_test)
y_pred_train = model.predict(X_train)
y_pred_proba = model.predict_proba(X_test)

C:\Users\rah\AppData\Local\Temp\ipykernel_20700\3480322820.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_prepared['age'].fillna(X_prepared['age'].median(), inplace=True)
C:\Users\rah\AppData\Local\Temp\ipykernel_20700\3480322820.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as 

In [61]:
target_names = ['not survived', 'survived']

print("\n" + "="*60)
print("MODEL EVALUATION")
print("="*60)

# accuracy
test_accuracy = accuracy_score(y_test, y_pred)
train_accuracy = accuracy_score(y_train, y_pred_train)
gap = train_accuracy - test_accuracy
print(f"Training Accuracy: {train_accuracy:.2%}")
print(f"Test Accuracy: {test_accuracy:.2%}")
print(f"Overfitting Gap: {gap:.4f}")


if gap < 0.01:
    print("✅ NO OVERFITTING (gap < 1%)")
    status = "Balanced"
elif gap < 0.02:
    print("✅ MINIMAL OVERFITTING (1-2%)")
    status = "Slight Overfitting"
elif gap < 0.03:
    print("⚠️  MILD OVERFITTING (2-3%)")
    status = "Mild Overfitting"
elif gap < 0.05:
    print("⚠️  MODERATE OVERFITTING (3-5%)")
    status = "Moderate Overfitting"
elif gap < 0.10:
    print("🔴 SIGNIFICANT OVERFITTING (5-10%)")
    status = "Significant Overfitting"
else:
    print("🔴 SEVERE OVERFITTING (>10%)")
    status = "Severe Overfitting"

if train_accuracy < 0.70 and test_accuracy < 0.70:
    print("📉 UNDERFITTING DETECTED: Model is too simple")
elif train_accuracy < 0.75:
    print("⚠️  Possible underfitting (training accuracy < 75%)")
else:
    print("✅ No underfitting detected")

# detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

# confusion matrix
cm = confusion_matrix(y_test, y_pred)
correct_predictions = np.trace(cm)
total_predictions = np.sum(cm)
accuracy = correct_predictions / total_predictions

print("\nConfusion Matrix:")
print(cm)


print(f"Correct predictions: {correct_predictions}")
print(f"Total predictions: {total_predictions}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy*100:.2f}%")
print()

for i in range(cm.shape[0]):
    class_correct = cm[i, i]
    class_total = np.sum(cm[i, :])
    accuracy = class_correct / class_total
    print(f"{target_names[i]} Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")


cv_scores = cross_val_score(model, X_train, y_train, cv=5)

print("\n" + "="*60)
print("CROSS-VALIDATION")
print("="*60)
print(f"CV Scores: {cv_scores}")
print(f"CV Mean: {cv_scores.mean():.4f}")
print(f"CV Std: {cv_scores.std():.4f}")


MODEL EVALUATION
Training Accuracy: 84.27%
Test Accuracy: 80.45%
Overfitting Gap: 0.0382
⚠️  MODERATE OVERFITTING (3-5%)
✅ No underfitting detected

Classification Report:
              precision    recall  f1-score   support

not survived       0.80      0.92      0.85       110
    survived       0.83      0.62      0.71        69

    accuracy                           0.80       179
   macro avg       0.81      0.77      0.78       179
weighted avg       0.81      0.80      0.80       179


Confusion Matrix:
[[101   9]
 [ 26  43]]
Correct predictions: 144
Total predictions: 179
Accuracy: 0.8045
Accuracy: 80.45%

not survived Accuracy: 0.9182 (91.82%)
survived Accuracy: 0.6232 (62.32%)

CROSS-VALIDATION
CV Scores: [0.7972028  0.76223776 0.86619718 0.83098592 0.83098592]
CV Mean: 0.8175
CV Std: 0.0352
